In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

import os
import pandas as pd

ROOT = "/kaggle/input"

for current, dirs, files in os.walk(ROOT):
    depth = current.replace(ROOT, "").count(os.sep)
    if depth > 3:
        dirs[:] = []
        continue

    for file in files:
        full_path = os.path.join(current, file)
        if file.lower().endswith(".csv"):
            print(f"\nCSV: {full_path}")
            try:
                df = pd.read_csv(full_path, nrows=5)
                print("列名：", df.columns.tolist())
                print(df.head(3).to_string(index=False))
            except Exception as e:
                print("读取失败：", e)
        elif file.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            print("图片示例：", full_path)
            # 只打印一个示例，避免输出过长
            break


In [ ]:

import os
import pandas as pd

ROOT = "/kaggle/input/competitions/2026task2"

for current, dirs, files in os.walk(ROOT):
    depth = current.replace(ROOT, "").count(os.sep)

    if depth > 3:
        dirs[:] = []
        continue

    print(f"\n目录：{current}")

    csv_files = [f for f in files if f.lower().endswith(".csv")]
    image_files = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
    ]

    for file in csv_files:
        path = os.path.join(current, file)
        print(f"\nCSV 文件：{path}")

        try:
            df = pd.read_csv(path, nrows=5)
            print("列名：", df.columns.tolist())
            print(df.head(3).to_string(index=False))
        except Exception as e:
            print("读取失败：", e)

    if image_files:
        print("图片数量（当前目录直接包含）：", len(image_files))
        print("图片示例：", image_files[:5])


In [ ]:

import os
import pandas as pd
from collections import Counter

ROOT = "/kaggle/input/competitions/2026task2/fer_data/fer_data"
TRAIN_DIR = os.path.join(ROOT, "train")
TEST_DIR = os.path.join(ROOT, "test")
SUBMISSION_PATH = "/kaggle/input/competitions/2026task2/submission.csv"

# 1. 查看训练集的目录结构
print("训练集目录是否存在：", os.path.isdir(TRAIN_DIR))
print("\n训练集第一层内容：")

for name in sorted(os.listdir(TRAIN_DIR)):
    path = os.path.join(TRAIN_DIR, name)

    if os.path.isdir(path):
        count = sum(
            1 for f in os.listdir(path)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
        )
        print(f"类别文件夹：{name:<20} 图片数量：{count}")
    else:
        print("文件：", name)

# 2. 递归统计所有训练图片
print("\n训练集图片总数：")
train_count = 0

for current, dirs, files in os.walk(TRAIN_DIR):
    n = sum(
        f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
        for f in files
    )
    train_count += n

print(train_count)

# 3. 检查测试集及提交模板
test_files = {
    f for f in os.listdir(TEST_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
}

sub = pd.read_csv(SUBMISSION_PATH)
submission_ids = set(sub["ID"].astype(str))

print("\n测试图片数量：", len(test_files))
print("提交模板行数：", len(sub))
print("模板列名：", sub.columns.tolist())
print("测试图片中不在模板里的数量：", len(test_files - submission_ids))
print("模板中找不到对应测试图片的数量：", len(submission_ids - test_files))

print("\nEmotion 当前模板取值分布：")
print(sub["Emotion"].value_counts(dropna=False))


In [ ]:

import os
import torch
import torchvision
from torchvision.datasets import ImageFolder

ROOT = "/kaggle/input/competitions/2026task2/fer_data/fer_data"
TRAIN_DIR = os.path.join(ROOT, "train")

# 1. 检查 GPU
print("PyTorch 版本：", torch.__version__)
print("Torchvision 版本：", torchvision.__version__)
print("GPU 是否可用：", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 型号：", torch.cuda.get_device_name(0))

# 2. 检查训练集的类别与数字映射
dataset = ImageFolder(TRAIN_DIR)

print("\n类别名称：", dataset.classes)
print("类别数字映射：", dataset.class_to_idx)
print("训练图片总数：", len(dataset))

# 3. 检查一张图片是否能正常读取
image, label = dataset[0]
print("\n图片尺寸：", image.size)
print("图片对应的类别：", dataset.classes[label])

# 4. 查看数据集目录中有没有标签说明文件
for folder in [
    "/kaggle/input/competitions/2026task2",
    ROOT
]:
    print(f"\n目录 {folder} 中的文件：")
    for name in sorted(os.listdir(folder)):
        if not os.path.isdir(os.path.join(folder, name)):
            print(" ", name)


In [ ]:

%pip install -q transformers==4.57.1


In [ ]:

import torch
import transformers
from transformers import AutoImageProcessor, ViTForImageClassification

MODEL_NAME = "HardlyHumans/Facial-expression-detection"

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU 可用:", torch.cuda.is_available())

print("\n正在加载图像处理器...")
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
print("图像处理器加载成功")
print("输入尺寸配置:", processor.size)

print("\n正在检查模型配置...")
from transformers import AutoConfig
config = AutoConfig.from_pretrained(MODEL_NAME)
print("模型架构:", config.architectures)
print("模型类型:", config.model_type)
print("原始标签数量:", config.num_labels)
print("原始标签映射:", config.id2label)

print("\n模型文件检查完成！")


In [ ]:

import torch
from transformers import (
    AutoImageProcessor,
    ViTForImageClassification
)

MODEL_NAME = "HardlyHumans/Facial-expression-detection"

classes = [
    "Angry",
    "Fear",
    "Happy",
    "Neutral",
    "Sad",
    "Surprise"
]

id2label = {i: name for i, name in enumerate(classes)}
label2id = {name: i for i, name in id2label.items()}

print("正在加载图像处理器...")
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

print("\n正在加载六分类 ViT...")
model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

print("\n模型加载成功！")
print("类别数量:", model.config.num_labels)
print("标签映射:", model.config.id2label)
print("分类头输出维度:", model.classifier.out_features)

model = model.cuda()
print("模型已成功放到 GPU:", next(model.parameters()).device)


In [ ]:

import os
import random
import numpy as np
import torch
import torch.nn.functional as F

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from transformers import AutoImageProcessor, ViTForImageClassification

# =========================
# 1. 参数设置
# =========================
SEED = 42
ROOT = "/kaggle/input/competitions/2026task2/fer_data/fer_data"
TRAIN_DIR = os.path.join(ROOT, "train")
MODEL_NAME = "HardlyHumans/Facial-expression-detection"
BEST_PATH = "/kaggle/working/best_vit_model.pth"

BATCH_SIZE = 16
EPOCHS = 6
NUM_WORKERS = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "请先在 Kaggle 设置 GPU 加速器"

# =========================
# 2. 读取模型的图像归一化配置
# =========================
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
mean, std = processor.image_mean, processor.image_std

train_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

val_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

train_all = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_all = datasets.ImageFolder(TRAIN_DIR, transform=val_transform)

classes = train_all.classes
num_classes = len(classes)
labels = np.array([label for _, label in train_all.samples])
indices = np.arange(len(train_all))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.15,
    random_state=SEED,
    stratify=labels
)

train_set = Subset(train_all, train_idx)
val_set = Subset(val_all, val_idx)

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("类别顺序:", classes)
print("训练图片:", len(train_set))
print("验证图片:", len(val_set))
print("GPU:", torch.cuda.get_device_name(0))

# =========================
# 3. 加载六分类模型
# =========================
id2label = {i: name for i, name in enumerate(classes)}
label2id = {name: i for i, name in id2label.items()}

model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(device)

# =========================
# 4. 优化器与混合精度
# =========================
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

scaler = torch.amp.GradScaler("cuda")
best_acc = 0.0

# =========================
# 5. 开始训练
# =========================
for epoch in range(EPOCHS):
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for step, (images, targets) in enumerate(train_loader, start=1):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(pixel_values=images).logits
            loss = F.cross_entropy(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        n = targets.size(0)
        train_loss_sum += loss.item() * n
        train_correct += (logits.argmax(1) == targets).sum().item()
        train_total += n

        if step % 100 == 0:
            print(
                f"Epoch {epoch+1}/{EPOCHS} "
                f"Step {step}/{len(train_loader)} "
                f"Loss {loss.item():.4f}"
            )

    # =========================
    # 6. 验证
    # =========================
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(pixel_values=images).logits
                loss = F.cross_entropy(logits, targets)

            n = targets.size(0)
            val_loss_sum += loss.item() * n
            val_correct += (logits.argmax(1) == targets).sum().item()
            val_total += n

    scheduler.step()

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total
    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    print("\n" + "=" * 60)
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4%}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4%}")
    print("=" * 60)

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "classes": classes,
            "class_to_idx": train_all.class_to_idx,
            "best_val_acc": best_acc,
            "epoch": epoch + 1,
        }, BEST_PATH)
        print("保存当前最佳模型:", BEST_PATH)

print("\n训练结束")
print(f"最佳验证准确率: {best_acc:.4%}")
print("最佳模型:", BEST_PATH)


In [ ]:

import os
import numpy as np
import torch
import pandas as pd

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoImageProcessor, ViTForImageClassification

ROOT = "/kaggle/input/competitions/2026task2/fer_data/fer_data"
TRAIN_DIR = os.path.join(ROOT, "train")
BEST_PATH = "/kaggle/working/best_vit_model.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "HardlyHumans/Facial-expression-detection"

# 使用和训练时相同的验证集划分
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
val_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=processor.image_mean,
        std=processor.image_std
    ),
])

dataset = datasets.ImageFolder(TRAIN_DIR, transform=val_transform)
classes = dataset.classes
labels = np.array([label for _, label in dataset.samples])
indices = np.arange(len(dataset))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.15,
    random_state=42,
    stratify=labels
)

val_loader = DataLoader(
    Subset(dataset, val_idx),
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# 加载最佳模型权重
checkpoint = torch.load(
    BEST_PATH,
    map_location="cpu",
    weights_only=True
)

model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(classes),
    id2label={i: name for i, name in enumerate(classes)},
    label2id={name: i for i, name in enumerate(classes)},
    ignore_mismatched_sizes=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, targets in val_loader:
        images = images.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = model(pixel_values=images).logits

        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(targets.numpy())

print("最佳模型轮次:", checkpoint["epoch"])
print("保存的最佳验证准确率:", f"{checkpoint['best_val_acc']:.4%}")

print("\n各类别表现：")
print(classification_report(
    all_labels,
    all_preds,
    labels=list(range(len(classes))),
    target_names=classes,
    digits=4,
    zero_division=0
))

cm = confusion_matrix(
    all_labels,
    all_preds,
    labels=list(range(len(classes)))
)

print("\n混淆矩阵（行=真实类别，列=预测类别）：")
print(pd.DataFrame(
    cm,
    index=[f"真实_{c}" for c in classes],
    columns=[f"预测_{c}" for c in classes]
))


In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm

# =========================
# 1. 固定随机种子
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
# 2. 数据路径
# =========================
DATA_DIR = "/kaggle/input/competitions/2026task2/fer_data/fer_data/train"

# =========================
# 3. 数据增强与预处理
# =========================
IMG_SIZE = 96
BATCH_SIZE = 64
NUM_WORKERS = 2

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 用两个数据集共享同一套文件索引，但使用不同的图像变换
train_dataset_aug = datasets.ImageFolder(DATA_DIR, transform=train_transform)
val_dataset_plain = datasets.ImageFolder(DATA_DIR, transform=val_transform)

assert train_dataset_aug.classes == val_dataset_plain.classes

classes = train_dataset_aug.classes
targets = np.array(train_dataset_aug.targets)
indices = np.arange(len(targets))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.15,
    random_state=SEED,
    stratify=targets
)

train_dataset = Subset(train_dataset_aug, train_idx)
val_dataset = Subset(val_dataset_plain, val_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Classes:", train_dataset_aug.class_to_idx)
print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
# =========================
# 1. 加载 ImageNet 预训练 ResNet18
# =========================
weights = ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# 替换最后的分类层
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, 6)
)

model = model.to(device)

# =========================
# 2. 损失函数与优化器
# =========================
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW([
    {
        "params": [p for name, p in model.named_parameters()
                   if not name.startswith("fc.")],
        "lr": 1e-5
    },
    {
        "params": model.fc.parameters(),
        "lr": 1e-4
    }
], weight_decay=1e-4)

EPOCHS = 12

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

best_acc = 0.0
best_path = "/kaggle/working/best_resnet18.pth"

# =========================
# 3. 训练与验证
# =========================
for epoch in range(EPOCHS):
    model.train()
    train_correct = 0
    train_total = 0
    train_loss_sum = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss_sum += loss.item() * labels.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

        pbar.set_postfix(
            train_acc=f"{100*train_correct/train_total:.2f}%"
        )

    # 验证阶段
    model.eval()
    val_correct = 0
    val_total = 0
    val_loss_sum = 0.0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss_sum += loss.item() * labels.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    train_loss = train_loss_sum / train_total
    val_loss = val_loss_sum / val_total

    scheduler.step()

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc*100:.2f}%"
    )

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "classes": classes,
            "class_to_idx": train_dataset_aug.class_to_idx,
            "img_size": IMG_SIZE,
            "best_val_acc": best_acc,
            "seed": SEED
        }, best_path)
        print(f"已保存当前最佳模型：{best_path}")

print(f"\nResNet18 最佳验证准确率：{best_acc*100:.2f}%")
print("模型路径：", best_path)

In [ ]:
```python
import os

for path in [
    "/kaggle/working/best_vit_model.pth",
    "/kaggle/working/best_resnet18.pth",
    "/kaggle/working/best_vit_v2.pth",
]:
    print(path, "存在" if os.path.exists(path) else "不存在")
```

In [ ]:
  File "/tmp/ipykernel_49/2865621402.py", line 1
    ```python
    ^
SyntaxError: invalid syntax

In [ ]:
import os

for path in [
    "/kaggle/working/best_vit_model.pth",
    "/kaggle/working/best_resnet18.pth",
    "/kaggle/working/best_vit_v2.pth",
]:
    print(path, "存在" if os.path.exists(path) else "不存在")

In [ ]:
import os
import random
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "/kaggle/input/competitions/2026task2/fer_data/fer_data/train"
IMG_SIZE = 224
BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = datasets.ImageFolder(DATA_DIR, transform=val_transform)
targets = np.array(dataset.targets)
indices = np.arange(len(targets))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.15,
    random_state=SEED,
    stratify=targets
)

val_dataset = Subset(dataset, val_idx)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

print("类别映射:", dataset.class_to_idx)
print("验证集图片数:", len(val_dataset))
print("设备:", device)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from torchvision.models import vit_b_16
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# 1. 基本设置
MODEL_PATH = "/kaggle/working/best_vit_model.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("当前设备:", device)

# 2. 加载模型
checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu",
    weights_only=True
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

model = vit_b_16(weights=None)
model.heads.head = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.heads.head.in_features, 6)
)

model.load_state_dict(state_dict, strict=True)
model = model.to(device)
model.eval()

print("ViT 模型加载成功！")

# 3. 验证集预测
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device, non_blocking=True)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model(images)

        all_preds.extend(outputs.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.tolist())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# 4. 计算准确率
acc = accuracy_score(all_labels, all_preds)
print(f"\nViT 验证集准确率: {acc * 100:.2f}%")

# 5. 各类别表现
print("\n各类别分类报告：")
print(classification_report(
    all_labels,
    all_preds,
    labels=list(range(6)),
    target_names=dataset.classes,
    digits=4,
    zero_division=0
))

# 6. 绘制混淆矩阵
cm = confusion_matrix(all_labels, all_preds, labels=list(range(6)))

fig, ax = plt.subplots(figsize=(9, 8))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=dataset.classes
)
disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
plt.title("ViT Confusion Matrix")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

# 7. 输出各类别最常见的错误
print("\n各类别最容易混淆的表情：")
for i, name in enumerate(dataset.classes):
    row = cm[i].copy()
    row[i] = 0
    j = int(np.argmax(row))
    print(f"{name} 被误判为 {dataset.classes[j]}：{row[j]} 张")

print("\n分析完成！")

In [ ]:

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from transformers import ViTForImageClassification
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# 1. 基本设置
MODEL_PATH = "/kaggle/working/best_vit_model.pth"
MODEL_NAME = "HardlyHumans/Facial-expression-detection"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("当前设备:", device)

# 2. 检查前面是否已经创建验证集 val_loader
if "val_loader" not in globals():
    raise NameError(
        "找不到 val_loader。请先运行之前创建验证集和 val_loader 的代码单元格。"
    )

# 3. 创建正确的 Hugging Face ViT 架构
model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6,
    ignore_mismatched_sizes=True
)

# 4. 加载训练好的六分类模型权重
checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu",
    weights_only=True
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

# 如果保存时使用了 DataParallel，去掉可能存在的 module. 前缀
if state_dict and all(k.startswith("module.") for k in state_dict):
    state_dict = {
        k[len("module."):]: v
        for k, v in state_dict.items()
    }

model.load_state_dict(state_dict, strict=True)
model = model.to(device)
model.eval()

print("模型权重加载成功！")

# 5. 在验证集上预测
all_preds = []
all_labels = []

with torch.inference_mode():
    for images, labels in val_loader:
        images = images.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model(pixel_values=images)
            preds = outputs.logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# 6. 输出准确率和分类报告
class_names = ["Angry", "Fear", "Happy", "Neutral", "Sad", "Surprise"]

acc = accuracy_score(all_labels, all_preds)
print(f"\n验证集准确率: {acc:.4%}")
print("\n各类别详细指标：")
print(
    classification_report(
        all_labels,
        all_preds,
        labels=list(range(6)),
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

# 7. 绘制混淆矩阵
cm = confusion_matrix(all_labels, all_preds, labels=list(range(6)))

fig, ax = plt.subplots(figsize=(9, 8))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)
disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    xticks_rotation=30,
    colorbar=False
)
ax.set_title(f"ViT Confusion Matrix (Accuracy: {acc:.2%})")
plt.tight_layout()
plt.show()

# 8. 列出最常见的分类错误
errors = []
for true_id in range(6):
    for pred_id in range(6):
        if true_id != pred_id and cm[true_id, pred_id] > 0:
            errors.append(
                (
                    int(cm[true_id, pred_id]),
                    class_names[true_id],
                    class_names[pred_id]
                )
            )

errors.sort(reverse=True)

print("\n最常见的 10 种分类错误：")
for count, true_name, pred_name in errors[:10]:
    print(f"真实类别 {true_name} -> 预测为 {pred_name}: {count} 张")


In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import torch
from collections import defaultdict

class_names = ["Angry", "Fear", "Happy", "Neutral", "Sad", "Surprise"]

model.eval()
wrong_images = defaultdict(list)

with torch.inference_mode():
    for images, labels in val_loader:
        images_device = images.to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = model(pixel_values=images_device).logits

        preds = logits.argmax(dim=1).cpu().numpy()
        labels_np = labels.numpy()

        for i, (true_id, pred_id) in enumerate(zip(labels_np, preds)):
            if true_id != pred_id:
                key = (int(true_id), int(pred_id))
                if len(wrong_images[key]) < 8:
                    wrong_images[key].append(
                        (images[i].cpu(), int(true_id), int(pred_id))
                    )

pairs = [
    (3, 4),  # Neutral -> Sad
    (1, 4),  # Fear -> Sad
    (0, 4),  # Angry -> Sad
    (5, 1),  # Surprise -> Fear
]

for true_id, pred_id in pairs:
    samples = wrong_images[(true_id, pred_id)]
    if not samples:
        print(f"没有找到 {class_names[true_id]} -> {class_names[pred_id]} 的错误图片")
        continue

    fig, axes = plt.subplots(
        1, len(samples),
        figsize=(2.2 * len(samples), 3)
    )
    if len(samples) == 1:
        axes = [axes]

    for ax, (img, true_label, pred_label) in zip(axes, samples):
        # 当前验证变换使用了 ImageNet 标准化，需要反标准化显示
        img = img.numpy().transpose(1, 2, 0)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = np.clip(img * std + mean, 0, 1)

        ax.imshow(img)
        ax.set_title(
            f"True: {class_names[true_label]}\nPred: {class_names[pred_label]}",
            fontsize=9
        )
        ax.axis("off")

    plt.suptitle(
        f"{class_names[true_id]} predicted as {class_names[pred_id]}"
    )
    plt.tight_layout()
    plt.show()


In [ ]:

import numpy as np
from sklearn.metrics import confusion_matrix

class_names = ["Angry", "Fear", "Happy", "Neutral", "Sad", "Surprise"]

# 使用前面已经得到的 all_labels 和 all_preds
cm = confusion_matrix(all_labels, all_preds, labels=list(range(6)))

print(f"{'类别':<12}{'真实数量':>10}{'预测数量':>10}{'正确数量':>10}{'召回率':>12}")
print("-" * 54)

for i, name in enumerate(class_names):
    true_count = int(cm[i].sum())
    pred_count = int(cm[:, i].sum())
    correct_count = int(cm[i, i])
    recall = correct_count / true_count if true_count else 0

    print(
        f"{name:<12}{true_count:>10}{pred_count:>10}"
        f"{correct_count:>10}{recall:>11.2%}"
    )

print("\n预测为 Sad 的图片中，各真实类别分别有多少张：")
for i, name in enumerate(class_names):
    print(f"真实类别 {name}: {int(cm[i, 4])} 张")

print(f"\n总计预测为 Sad: {int(cm[:, 4].sum())} 张")


In [ ]:

import os
import copy
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from transformers import ViTForImageClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm

# ============================================================
# 1. 配置
# ============================================================
SEED = 42
DATA_DIR = "/kaggle/input/competitions/2026task2/fer_data/fer_data/train"
MODEL_NAME = "HardlyHumans/Facial-expression-detection"
BASE_MODEL_PATH = "/kaggle/working/best_vit_model.pth"
SAVE_PATH = "/kaggle/working/best_vit_weighted.pth"

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
BACKBONE_LR = 5e-6
HEAD_LR = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("设备:", device)

if not os.path.isfile(BASE_MODEL_PATH):
    raise FileNotFoundError(
        f"找不到原模型：{BASE_MODEL_PATH}\n"
        "请确认 best_vit_model.pth 仍然存在。"
    )

# ============================================================
# 2. 数据增强与验证集
# 保持验证集划分与之前一致
# ============================================================
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=8),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 同一份图片列表，训练集和验证集使用不同变换
train_dataset_all = datasets.ImageFolder(DATA_DIR, transform=train_transform)
val_dataset_all = datasets.ImageFolder(DATA_DIR, transform=val_transform)

class_names = train_dataset_all.classes
print("类别映射:", train_dataset_all.class_to_idx)
print("类别顺序:", class_names)

targets = np.array(train_dataset_all.targets)
indices = np.arange(len(targets))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.15,
    random_state=SEED,
    stratify=targets
)

train_dataset = Subset(train_dataset_all, train_idx)
val_dataset = Subset(val_dataset_all, val_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

print("训练图片:", len(train_dataset))
print("验证图片:", len(val_dataset))

# ============================================================
# 3. 加载原始 ViT 模型及权重
# ============================================================
model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6,
    ignore_mismatched_sizes=True
)

checkpoint = torch.load(
    BASE_MODEL_PATH,
    map_location="cpu",
    weights_only=True
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

if state_dict and all(k.startswith("module.") for k in state_dict):
    state_dict = {
        k[len("module."):]: v for k, v in state_dict.items()
    }

model.load_state_dict(state_dict, strict=True)
model = model.to(device)
print("原始模型权重加载成功")

# ============================================================
# 4. 类别加权损失
# 类别顺序：Angry, Fear, Happy, Neutral, Sad, Surprise
# ============================================================
# 适度提高 Angry、Fear、Neutral 的权重，降低 Sad 的相对权重。
# 不采用极端权重，避免小类别被过度补偿。
class_weights = torch.tensor(
    [1.20, 1.25, 1.00, 1.20, 0.80, 1.05],
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# ============================================================
# 5. 分组学习率
# 预训练骨干使用较小学习率，分类头使用较大学习率
# ============================================================
head_params = list(model.classifier.parameters())
head_param_ids = {id(p) for p in head_params}
backbone_params = [
    p for p in model.parameters()
    if id(p) not in head_param_ids
]

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": head_params, "lr": HEAD_LR}
    ],
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

# ============================================================
# 6. 先评估原模型，确认基线
# ============================================================
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0.0
    total_num = 0

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                logits = model(pixel_values=images).logits
                loss = nn.functional.cross_entropy(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total_num += labels.size(0)

            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    avg_loss = total_loss / max(total_num, 1)
    return avg_loss, acc, np.array(all_labels), np.array(all_preds)

base_loss, base_acc, base_labels, base_preds = evaluate(model, val_loader)

print(f"\n原始模型验证准确率: {base_acc:.4%}")
print(f"原始模型验证损失: {base_loss:.4f}")

# 以原模型作为初始最佳模型。
# 新模型只有准确率更高时才覆盖实验结果文件。
best_acc = base_acc
best_state = copy.deepcopy(model.state_dict())
best_epoch = 0
epochs_without_improvement = 0

# ============================================================
# 7. 微调
# ============================================================
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    total = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for images, labels in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = model(pixel_values=images).logits
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * labels.size(0)
        total += labels.size(0)

        progress.set_postfix(loss=f"{loss.item():.4f}")

    scheduler.step()

    train_loss = running_loss / max(total, 1)
    val_loss, val_acc, y_true, y_pred = evaluate(model, val_loader)

    print(
        f"\nEpoch {epoch + 1}: "
        f"Train Loss={train_loss:.4f} | "
        f"Val Loss={val_loss:.4f} | "
        f"Val Acc={val_acc:.4%}"
    )

    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0

        torch.save(best_state, SAVE_PATH)
        print("验证准确率提高，已保存:", SAVE_PATH)
    else:
        epochs_without_improvement += 1
        print("验证准确率没有超过历史最佳。")

    if epochs_without_improvement >= PATIENCE:
        print("连续多轮没有提升，提前停止。")
        break

# ============================================================
# 8. 汇报最佳结果
# ============================================================
print("\n" + "=" * 60)
print(f"原始模型准确率: {base_acc:.4%}")
print(f"实验最佳准确率: {best_acc:.4%}")
print(f"实验最佳轮次: {best_epoch}")

if best_acc > base_acc:
    print(f"提升: {(best_acc - base_acc) * 100:.2f} 个百分点")
    print("最佳实验权重:", SAVE_PATH)
else:
    print("本次实验未超过原始模型，继续使用原模型更稳妥。")

# 使用最佳权重重新输出分类报告
model.load_state_dict(best_state, strict=True)
_, final_acc, y_true, y_pred = evaluate(model, val_loader)

print(f"\n最佳模型验证准确率: {final_acc:.4%}")
print("\n最佳模型分类报告:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(6)),
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


In [ ]:

import os
import shutil

model_path = "/kaggle/working/best_vit_weighted.pth"
backup_path = "/kaggle/working/best_vit_weighted_backup.pth"

if not os.path.exists(model_path):
    raise FileNotFoundError(
        "没有找到 best_vit_weighted.pth，请先检查之前的训练输出。"
    )

shutil.copy2(model_path, backup_path)

print("模型文件存在：", os.path.exists(model_path))
print("备份文件存在：", os.path.exists(backup_path))
print("模型大小：", round(os.path.getsize(model_path) / 1024 / 1024, 2), "MB")
print("备份完成！原模型没有被修改。")


In [ ]:

import os
import copy
import random
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from transformers import ViTForImageClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm

# ==================== 1. 配置 ====================
SEED = 42
DATA_DIR = "/kaggle/input/competitions/2026task2/fer_data/fer_data/train"
MODEL_NAME = "HardlyHumans/Facial-expression-detection"

BASE_PATH = "/kaggle/working/best_vit_weighted.pth"
SAVE_PATH = "/kaggle/working/best_vit_weighted_v2.pth"

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 4
BACKBONE_LR = 1e-6
HEAD_LR = 5e-6
WEIGHT_DECAY = 0.01
PATIENCE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("设备:", device)

if not os.path.isfile(BASE_PATH):
    raise FileNotFoundError(f"找不到起始模型：{BASE_PATH}")

# ==================== 2. 数据集 ====================
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=8),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_all = datasets.ImageFolder(DATA_DIR, transform=train_transform)
val_all = datasets.ImageFolder(DATA_DIR, transform=val_transform)

class_names = train_all.classes
targets = np.array(train_all.targets)
indices = np.arange(len(targets))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.15,
    random_state=SEED,
    stratify=targets
)

train_dataset = Subset(train_all, train_idx)
val_dataset = Subset(val_all, val_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

print("类别:", class_names)
print("训练集:", len(train_dataset), "验证集:", len(val_dataset))

# ==================== 3. 加载上一轮最佳模型 ====================
model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6,
    ignore_mismatched_sizes=True
)

state = torch.load(BASE_PATH, map_location="cpu", weights_only=True)
if isinstance(state, dict) and "model_state_dict" in state:
    state = state["model_state_dict"]

if state and all(k.startswith("module.") for k in state):
    state = {k[7:]: v for k, v in state.items()}

model.load_state_dict(state, strict=True)
model = model.to(device)
print("已加载上一轮 70.89% 模型")

# ==================== 4. 损失函数和优化器 ====================
class_weights = torch.tensor(
    [1.20, 1.25, 1.00, 1.20, 0.80, 1.05],
    dtype=torch.float32,
    device=device
)
criterion = nn.CrossEntropyLoss(weight=class_weights)

head_params = list(model.classifier.parameters())
head_ids = {id(p) for p in head_params}
backbone_params = [
    p for p in model.parameters() if id(p) not in head_ids
]

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": head_params, "lr": HEAD_LR}
    ],
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

scaler = torch.amp.GradScaler(
    "cuda", enabled=(device.type == "cuda")
)

# ==================== 5. 验证函数 ====================
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss, total = 0.0, 0

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                logits = model(pixel_values=images).logits
                loss = nn.functional.cross_entropy(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total += labels.size(0)
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return (
        total_loss / total,
        accuracy_score(all_labels, all_preds),
        np.array(all_labels),
        np.array(all_preds)
    )

# ==================== 6. 评估起始模型 ====================
base_loss, base_acc, _, _ = evaluate(model, val_loader)
print(f"\n起始模型验证准确率: {base_acc:.4%}")

best_acc = base_acc
best_state = copy.deepcopy(model.state_dict())
best_epoch = 0
no_improve = 0

# ==================== 7. 微调 ====================
for epoch in range(EPOCHS):
    model.train()
    total_loss, total = 0.0, 0

    for images, labels in tqdm(
        train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}"
    ):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = model(pixel_values=images).logits
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * labels.size(0)
        total += labels.size(0)

    scheduler.step()
    train_loss = total_loss / total
    val_loss, val_acc, y_true, y_pred = evaluate(model, val_loader)

    print(
        f"\nEpoch {epoch + 1}: "
        f"Train Loss={train_loss:.4f} | "
        f"Val Loss={val_loss:.4f} | "
        f"Val Acc={val_acc:.4%}"
    )

    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
        torch.save(best_state, SAVE_PATH)
        print("新最佳权重已保存:", SAVE_PATH)
    else:
        no_improve += 1
        print("本轮没有超过历史最佳。")

    if no_improve >= PATIENCE:
        print("连续两轮没有提升，提前停止。")
        break

# ==================== 8. 输出最终结果 ====================
model.load_state_dict(best_state, strict=True)
_, final_acc, y_true, y_pred = evaluate(model, val_loader)

print("\n" + "=" * 55)
print(f"起始模型准确率: {base_acc:.4%}")
print(f"本轮最佳准确率: {best_acc:.4%}")
print(f"最佳轮次: {best_epoch}")

if best_acc > base_acc:
    print(f"相对提升: {(best_acc - base_acc) * 100:.2f} 个百分点")
    print("新模型路径:", SAVE_PATH)
else:
    print("本轮没有提升，继续使用 best_vit_weighted.pth")

print("\n分类报告:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(6)),
    target_names=class_names,
    digits=4,
    zero_division=0
))


In [ ]:

import os
import pandas as pd

# 官方提交模板
sub_path = "/kaggle/input/competitions/2026task2/submission.csv"
sub = pd.read_csv(sub_path)

print("===== 官方提交模板 =====")
print("形状：", sub.shape)
print("列名：", sub.columns.tolist())
print("前10行：")
print(sub.head(10).to_string(index=False))
print("\nEmotion 列的数据类型：", sub["Emotion"].dtype)
print("Emotion 列现有值：")
print(sub["Emotion"].value_counts(dropna=False).sort_index())

# 测试集目录
test_dir = "/kaggle/input/competitions/2026task2/fer_data/fer_data/test"

print("\n===== 测试集文件 =====")
files = sorted([
    f for f in os.listdir(test_dir)
    if os.path.isfile(os.path.join(test_dir, f))
])

print("测试图片数量：", len(files))
print("前20个文件名：")
print(files[:20])

print("\n===== 当前最佳模型 =====")
for p in [
    "/kaggle/working/best_vit_weighted.pth",
    "/kaggle/working/best_vit_weighted_backup.pth",
]:
    print(p, "存在：", os.path.exists(p))
    if os.path.exists(p):
        print("文件大小：", round(os.path.getsize(p) / 1024**2, 2), "MB")


In [ ]:

import os
import pandas as pd
from torchvision import datasets

train_dir = "/kaggle/input/competitions/2026task2/fer_data/fer_data/train"
test_dir = "/kaggle/input/competitions/2026task2/fer_data/fer_data/test"
sub_path = "/kaggle/input/competitions/2026task2/submission.csv"

# 检查训练集类别与 ImageFolder 编码
train_data = datasets.ImageFolder(train_dir)
print("训练集类别编码：", train_data.class_to_idx)
print("训练集图片数量：", len(train_data))

# 检查模板中的 ID 是否都存在于测试集
sub = pd.read_csv(sub_path)
test_files = set(os.listdir(test_dir))
template_ids = set(sub["ID"].astype(str))

print("模板行数：", len(sub))
print("测试目录文件数：", len(test_files))
print("模板 ID 在测试目录中全部存在：", template_ids.issubset(test_files))
print("模板中重复 ID 数量：", sub["ID"].duplicated().sum())

# 检查训练集是否存在类别名称与编码以外的标签说明文件
for root, dirs, files in os.walk("/kaggle/input/competitions/2026task2"):
    for f in files:
        if f.lower() in [
            "label_map.json", "labels.json", "classes.txt",
            "class_to_idx.json", "readme.md"
        ]:
            print("可能的标签说明文件：", os.path.join(root, f))


In [ ]:

import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import ViTForImageClassification

# =========================
# 1. 路径和配置
# =========================
DATA_ROOT = "/kaggle/input/competitions/2026task2/fer_data/fer_data"
TEST_DIR = os.path.join(DATA_ROOT, "test")
TEMPLATE_PATH = "/kaggle/input/competitions/2026task2/submission.csv"

MODEL_PATH = "/kaggle/working/best_vit_weighted.pth"
OUTPUT_PATH = "/kaggle/working/submission.csv"

BATCH_SIZE = 32
NUM_WORKERS = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用设备：", device)

assert os.path.exists(MODEL_PATH), f"找不到模型：{MODEL_PATH}"

# =========================
# 2. 读取官方模板
# =========================
sub = pd.read_csv(TEMPLATE_PATH)

assert sub.columns.tolist() == ["ID", "Emotion"], "提交模板列名不符合预期"
assert sub["ID"].is_unique, "模板中存在重复 ID"

ids = sub["ID"].astype(str).tolist()

for image_id in ids:
    image_path = os.path.join(TEST_DIR, image_id)
    if not os.path.isfile(image_path):
        raise FileNotFoundError(f"测试图片不存在：{image_path}")

print("待预测图片数量：", len(ids))

# =========================
# 3. 测试集预处理
# 必须与验证时保持一致
# =========================
test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class FERTestDataset(Dataset):
    def __init__(self, image_ids, image_dir, transform):
        self.image_ids = image_ids
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_path = os.path.join(self.image_dir, image_id)

        with Image.open(image_path) as im:
            image = im.convert("L")
            image = self.transform(image)

        return image

test_dataset = FERTestDataset(ids, TEST_DIR, test_transform)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

# =========================
# 4. 加载六分类 ViT 模型
# =========================
MODEL_NAME = "HardlyHumans/Facial-expression-detection"

model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6,
    ignore_mismatched_sizes=True
)

state = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
model.load_state_dict(state)
model.to(device)
model.eval()

print("模型加载成功")

# =========================
# 5. 执行预测
# =========================
predictions = []

with torch.inference_mode():
    for images in tqdm(test_loader, desc="预测测试集"):
        images = images.to(device, non_blocking=True)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = model(pixel_values=images).logits

        batch_preds = logits.argmax(dim=1).cpu().numpy()
        predictions.extend(batch_preds.tolist())

assert len(predictions) == len(ids), (
    f"预测数量错误：{len(predictions)}，预期：{len(ids)}"
)

# =========================
# 6. 生成提交文件
# =========================
result = pd.DataFrame({
    "ID": ids,
    "Emotion": np.asarray(predictions, dtype=np.int64)
})

result.to_csv(OUTPUT_PATH, index=False)

# =========================
# 7. 检查输出
# =========================
check = pd.read_csv(OUTPUT_PATH)

print("\n===== 提交文件检查 =====")
print("保存位置：", OUTPUT_PATH)
print("文件大小：", round(os.path.getsize(OUTPUT_PATH) / 1024, 2), "KB")
print("行数：", len(check))
print("列名：", check.columns.tolist())
print("ID 顺序与官方模板一致：", check["ID"].tolist() == ids)
print("是否有空值：", check.isnull().any().any())
print("预测标签分布：")
print(check["Emotion"].value_counts().sort_index())
print("\n前10行：")
print(check.head(10).to_string(index=False))

assert len(check) == 7065
assert check.columns.tolist() == ["ID", "Emotion"]
assert check["ID"].tolist() == ids
assert not check.isnull().any().any()
assert check["Emotion"].between(0, 5).all()

print("\n检查通过！提交文件已经生成。")


In [ ]:

from IPython.display import FileLink, display

display(FileLink("/kaggle/working/submission.csv"))


In [ ]:
import pandas as pd

sub = pd.read_csv("/kaggle/working/submission.csv")

print(sub["Emotion"].value_counts())
print()
print(sub.head(20))

In [ ]:
import os

print(os.listdir("/kaggle/working"))